# Part 2 — Sensors in a Dynamic Positioning System

This notebook starts with the most important estimation idea: the vessel has a physical state, but the controller cannot read that state directly. The plant produces simulation **truth**, sensors produce imperfect **measurements**, and an observer estimates the state needed for feedback. It then explains what a real DP sensor system contains and demonstrates the `models.sensors.VesselSensors` interface used by `DPSimulatorPart2`.

Course reference: *Marine Cybernetics*, Sections 2.6.1 (position reference systems and sensors), 5.3 (sensor signal processing), and 7.2 (observer design for DP).

The notebook builds the measurement picture up in three steps: **Step 1** a calm vessel with sensor noise (pure measurement error), **Step 2** wave-induced motion with a perfect sensor (real motion that only *looks* like noise), **Step 3** both together — followed by why this makes an observer necessary.

## What exists on a real DP vessel?

A DP system does not normally receive one perfect vector called `eta`. Its measurement system is a network of instruments:

- **Position references:** GNSS/DGPS, hydroacoustic position reference (HPR), taut wire, laser, microwave or operation-specific references. They measure position in different frames and can have different update rates, bias, noise and failure modes.
- **Heading:** one or more gyrocompasses; dual-antenna GNSS may provide another heading source.
- **Motion/attitude:** VRU, MRU or IMU containing gyros and accelerometers. These provide roll, pitch, angular rate and specific force.
- **Environment:** anemometers for relative wind speed/direction; some vessels also use wave and current sensors.
- **Operation-specific signals:** for example riser angle, pipe tension, gangway state or draft.

A production system also performs range and variance checks, detects frozen or drifting signals, compensates sensor lever arms, transforms coordinate frames, weights/votes redundant measurements, and handles sensors entering or leaving the solution without discontinuities. The literature notes a typical minimum DP configuration of a position reference, gyrocompass, VRU and wind sensor, with redundancy increasing for higher equipment classes.

## Our deliberate simplification

The project plant is the **Gunnerus 3-DOF model**. It integrates North, East and heading together with body velocities surge, sway and yaw rate. The simulator expands these values to a common 6-DOF layout:

`eta = [N, E, D, phi, theta, psi]` and `nu = [u, v, w, p, q, r]`.

`VesselSensors` has a single switch: with noise **off** (the default, used by the mandatory simulations) it passes pose and velocity through unchanged; with noise **on** it adds Gaussian noise with fixed standard deviations representing a DP-grade sensor suite (GNSS position, gyrocompass heading). The noise levels are course parameters defined in `models/sensors.py` — they are not student tuning knobs. The model does **not** simulate satellite geometry, acoustic propagation, gyro drift, IMU integration, update-rate differences, latency, lever arms, correlated noise, redundancy, voting, dropouts or faults.

Only measured pose is passed to the Part 2 observer. Noisy velocity is logged solely to compare an observer estimate with simulation truth. This mirrors the assumption that vessel velocity is normally estimated, rather than treated as an ordinary feedback measurement.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import welch

# Works when launched from either the repository root or notebooks/.
root = Path.cwd()
if not (root / 'models').exists():
    root = root.parent
sys.path.insert(0, str(root))

from models.sensors import SensorConfig, VesselSensors
from part_2.config import default_thrusters_part_2
from models.waves import Waves
from simulation.plant import GunnerusPlant3DOF

print('Project root:', root)

## Sensors OFF and ON

Measurement noise is switched per simulation with `Part2SimConfig(use_sensor_noise=...)`, which the engine forwards to `VesselSensors` as `SensorConfig(noise=...)`. The mandatory simulations use noise **off** — ideal measurements. Enabling the noise is an **optional** part of the project: your observer then has to filter measurement noise on top of the wave-frequency motion.

The standard deviations used when noise is on are fixed course parameters, so results are comparable between groups. This notebook constructs both cases side by side.

In [ ]:
sensors_off = VesselSensors()                         # default: ideal measurements
sensors_on = VesselSensors(SensorConfig(noise=True))  # optional exercise: fixed noise levels
print('OFF adds no noise; configured stds still visible:', sensors_off.config.eta_std)
print('ON pose standard deviations [m, m, -, -, -, rad]: ', sensors_on.config.eta_std)
print('ON velocity standard deviations (validation only):', sensors_on.config.nu_std)

## Plant truth is not a measurement

`simulation/plant.py` contains the process plant used to generate vessel motion. In simulation we can inspect its exact numerical state, called **truth pose** and **truth velocity**. On a real vessel there is no instrument that returns this exact internal state. Each physical instrument returns a measurement affected by noise, bias, dynamics, placement and possible faults.

**Wind note:** wind would enter the plant at the same physical-load boundary as waves and would therefore change truth motion, not sensor noise. It is intentionally omitted here so this notebook stays focused on waves and measurements. Students implement and study wind separately in `part_2/wind.py`.

## Step 1 — Calm water: sensor noise in isolation

First, no environment at all. Nothing pushes the vessel, so the truth stays constant — every deviation you see in the measurement is **sensor error and nothing else**. With the noise switch off the sensor returns the truth exactly; with the switch on it scatters around it with the fixed course noise levels.

In [ ]:
dt = 0.1
thrusters = default_thrusters_part_2()
zero_thrust = np.zeros(len(thrusters))
initial_angles = np.array([th.alpha0 for th in thrusters])

calm_plant = GunnerusPlant3DOF(thrusters, dt=dt, thruster_dynamics=True)
calm = calm_plant.run(T=180.0, thrust_command=zero_thrust,
                      azimuth_command=initial_angles)
t = calm['t']
eta_calm, nu_calm = calm['eta'], calm['nu']

sensors_on.reset()  # restart the seeded noise sequence for reproducibility
calm_off = [sensors_off.measure(e, n) for e, n in zip(eta_calm, nu_calm)]
calm_on = [sensors_on.measure(e, n) for e, n in zip(eta_calm, nu_calm)]
eta_calm_ideal = np.array([m[0] for m in calm_off])
eta_calm_meas = np.array([m[0] for m in calm_on])

items = [(0, 'North [m]', 1.0), (1, 'East [m]', 1.0),
         (5, 'Heading [deg]', 180/np.pi)]
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for ax, (idx, label, scale) in zip(axes, items):
    ax.plot(t, scale*eta_calm[:, idx], lw=2, label='calm truth')
    ax.plot(t, scale*eta_calm_ideal[:, idx], '--',
            label='sensor noise OFF (default)')
    ax.plot(t, scale*eta_calm_meas[:, idx], '.', ms=2, alpha=0.5,
            label='sensor noise ON (fixed course levels)')
    ax.set_ylabel(label); ax.grid(alpha=0.3)
axes[0].legend(); axes[-1].set_xlabel('Time [s]')
fig.suptitle('Calm water: every deviation is sensor error')
fig.tight_layout()

## Check what the noise model actually produces

For enough samples, the measurement error should have approximately zero mean and the configured standard deviation. Gaussian samples are unbounded, so a standard deviation is not a hard error limit. Real sensor errors can also contain bias, drift, colored noise and outliers; those are intentionally absent here.

In [ ]:
eta_error = eta_calm_meas - eta_calm
eta_std = sensors_on.config.eta_std
labels = ['North [m]', 'East [m]', 'Heading [deg]']
for idx, label, scale, target in [(0, labels[0], 1, eta_std[0]), (1, labels[1], 1, eta_std[1]), (5, labels[2], 180/np.pi, np.rad2deg(eta_std[5]))]:
    error = scale * eta_error[:, idx]
    print(f'{label:14s} mean={error.mean(): .4f}, sample std={error.std(ddof=1):.4f}, configured std={target:.4f}')

## Heading is a circular measurement

Angles near +180° and −180° describe nearly the same heading. `VesselSensors` wraps measured heading to the interval `(-pi, pi]`, whether the noise switch is on or off. Observer innovations must also use a wrapped angle difference.

In [ ]:
wrap_demo = VesselSensors()
eta_edge = np.array([0, 0, 0, 0, 0, np.deg2rad(181.0)])
eta_m, _ = wrap_demo.measure(eta_edge, np.zeros(6))
print(f'{np.rad2deg(eta_m[5]):7.2f} deg')

## Reproducibility and observer boundary

With the noise switch off the sensor is deterministic. With the switch on, the noise generator is seeded (`SensorConfig.seed`) and `reset()` restarts the sequence, so repeated runs — and two different observers — see exactly the same measurements. In `DPSimulatorPart2`, the selected nonlinear passive observer or Kalman filter receives only `(t, dt, eta_measured, tau_est)`. The controller then receives the observer's `eta` and `nu` estimates.

In [ ]:
sensors_on.reset()
first_eta, _ = sensors_on.measure(eta_calm[0], nu_calm[0])
sensors_on.reset()
repeated_eta, _ = sensors_on.measure(eta_calm[0], nu_calm[0])
print('Identical after reset:', np.array_equal(first_eta, repeated_eta))

from part_2.observer import select_observer
for kind in ('nonlinear_passive', 'kalman'):
    observer = select_observer(kind)
    estimate = observer.step(0.0, dt, first_eta, np.zeros(6))
    print(kind, '-> eta', estimate.eta.shape, 'nu', estimate.nu.shape, 'bias', estimate.bias.shape)

## Step 2 — Add waves: the truth itself starts to "look noisy"

Now the real Part 2 wave model acts on the plant, and the sensor noise stays **off**. Everything in these plots is real physical motion, with two time scales: a slowly-varying drift (second-order wave loads — the part DP must counteract) and a wave-frequency oscillation (first-order loads, period ≈ Tp = 8 s — the part DP must *ignore*). On the full time axis the drift dominates, so the right column zooms into a 60-second window where the oscillation is visible.

This resolves an easy misunderstanding: even with **zero** sensor noise the measurement oscillates — because the vessel is physically responding to the waves, and a perfect sensor faithfully reports that motion.

In [ ]:
plant = GunnerusPlant3DOF(thrusters, dt=dt, thruster_dynamics=True)
# Reference wave discretization; 180 s holds 20 wave periods.
waves = Waves(hs=1.5, tp=8.0, direction=np.deg2rad(45), seed=123, n_components=20)
truth = plant.run(
    T=180.0, thrust_command=zero_thrust, azimuth_command=initial_angles,
    waves=waves,
)
eta_true = truth['eta']       # exact plant state: unavailable on a real vessel
nu_true = truth['nu']         # exact plant velocity: unavailable on a real vessel

measured_off = [sensors_off.measure(e, n) for e, n in zip(eta_true, nu_true)]
eta_ideal = np.array([m[0] for m in measured_off])
nu_ideal = np.array([m[1] for m in measured_off])

zoom = (t >= 120.0) & (t <= 180.0)
fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex='col')
for row, (idx, label, scale) in zip(axes, items):
    for ax, sel in zip(row, [slice(None), zoom]):
        ax.plot(t[sel], scale*eta_calm[sel, idx], color='0.7', lw=1.5,
                label='calm truth (Step 1)')
        ax.plot(t[sel], scale*eta_true[sel, idx], label='truth with waves')
        ax.plot(t[sel], scale*eta_ideal[sel, idx], '--',
                label='measurement, sensor noise OFF')
        ax.grid(alpha=0.3)
    row[0].set_ylabel(label)
axes[0][0].set_title('Full run: slow wave drift dominates')
axes[0][1].set_title('Zoom 120-180 s: wave-frequency oscillation')
axes[0][0].legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel('Time [s]')
fig.suptitle('With a PERFECT sensor the measurement still oscillates: it is real motion, not noise')
fig.tight_layout()

In [ ]:
# The same run, velocity states.  The wave loads make the TRUE body
# velocities oscillate at the wave frequency -- none of this is sensor noise,
# and none of these signals is measurable on the real vessel.
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
vel_items = [(0, 'surge u [m/s]', 1.0), (1, 'sway v [m/s]', 1.0),
             (5, 'yaw rate r [deg/s]', 180/np.pi)]
for ax, (idx, label, scale) in zip(axes, vel_items):
    ax.plot(t, scale*nu_true[:, idx], label='true velocity')
    ax.set_ylabel(label); ax.grid(alpha=0.3)
axes[0].legend(); axes[-1].set_xlabel('Time [s]')
fig.suptitle('True velocity states already oscillate at the wave frequency')
fig.tight_layout()

## Step 3 — Waves and sensor noise together

Finally both effects at once. The measurement now contains three layers: the slow wave drift and the wave-frequency motion (both physical, Step 2) plus sensor noise (Step 1). The default noise-off case contains only the first two; switching the noise on adds the third.

In [ ]:
sensors_on.reset()
measured_on = [sensors_on.measure(e, n) for e, n in zip(eta_true, nu_true)]
eta_measured = np.array([m[0] for m in measured_on])
nu_measured = np.array([m[1] for m in measured_on])

fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex='col')
for row, (idx, label, scale) in zip(axes, items):
    for ax, sel in zip(row, [slice(None), zoom]):
        ax.plot(t[sel], scale*eta_calm[sel, idx], color='0.7', lw=1.5,
                label='calm truth')
        ax.plot(t[sel], scale*eta_true[sel, idx], label='truth with waves')
        ax.plot(t[sel], scale*eta_measured[sel, idx], '.', ms=2, alpha=0.5,
                label='waves + sensor noise ON')
        ax.grid(alpha=0.3)
    row[0].set_ylabel(label)
axes[0][0].set_title('Full run')
axes[0][1].set_title('Zoom 120-180 s: motion + noise layers')
axes[0][0].legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel('Time [s]')
fig.suptitle('Three layers: wave drift + wave-frequency motion + sensor noise')
fig.tight_layout()

The overlay may make small differences difficult to read. Residuals expose the source of each difference:

- `wave effect = wave truth − calm truth` is physical vessel response.
- `sensor error = noisy measurement − wave truth` is measurement corruption.

An observer sees only the measured signal and known model/input information; truth and these residuals are available only for simulation evaluation.

In [ ]:
wave_effect = eta_true - eta_calm
sensor_error = eta_measured - eta_true
# Circular differences for heading.
wave_effect[:, 5] = (wave_effect[:, 5] + np.pi) % (2*np.pi) - np.pi
sensor_error[:, 5] = (sensor_error[:, 5] + np.pi) % (2*np.pi) - np.pi

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, (idx, label, scale) in zip(axes, items):
    ax.plot(t, scale*wave_effect[:, idx], label='physical wave effect')
    ax.plot(t, scale*sensor_error[:, idx], label='sensor error (noise ON)', alpha=0.75)
    ax.set_ylabel(label); ax.grid(alpha=0.3)
axes[0].legend(); axes[-1].set_xlabel('Time [s]')
fig.suptitle('Physical motion and measurement error are different signals')
fig.tight_layout()

# The frequency domain makes the same point in one picture: the wave effect is
# COLORED (energy near DC from the slow drift plus a hump around the wave
# frequencies), while the sensor error is WHITE (flat).  A DP observer must
# keep the near-DC part, reject the hump, and average out the flat floor.
fs = 1.0 / dt
fig, ax = plt.subplots(figsize=(9, 3.5))
for sig, lab in [(wave_effect[:, 1], 'physical wave effect (East)'),
                 (sensor_error[:, 1], 'sensor error, noise ON (East)')]:
    f, P = welch(sig, fs=fs, nperseg=512)
    ax.semilogy(2*np.pi*f, P / (2*np.pi), label=lab)
ax.axvline(2*np.pi/8.0, color='gray', ls=':', label=r'wave peak $\omega_p$')
ax.set(xlabel=r'$\omega$ [rad/s]', xlim=(0, 3.0),
       ylabel=r'PSD [m$^2$ s]',
       title='Wave-induced motion is colored; sensor noise is white')
ax.grid(alpha=0.3, which='both'); ax.legend(); fig.tight_layout()

## Read the signal chain correctly

The example contains two separate transformations:

`waves → BODY forces/moment → plant dynamics → truth pose → sensors → measured pose`

Waves physically move the simulated vessel, changing truth pose and velocity. Turning sensor noise ON cannot move the vessel; it changes only the reported measurement. Both sensor cases therefore share exactly the same truth. With sensors OFF, measurement and truth overlap. With sensors ON, measurements scatter around that truth. Wind would behave like waves on this side of the sensor boundary: it changes vessel truth through a load, but does not create measurement noise.

In `DPSimulatorPart2`, the engine evaluates environmental loads using the true plant state, integrates Gunnerus, samples the resulting state through `VesselSensors`, and sends measured pose—not truth—to the observer. The controller uses the observer estimate.

In [ ]:
print('OFF measurement equals truth:', np.array_equal(eta_ideal, eta_true))
print('ON measurement equals truth: ', np.array_equal(eta_measured, eta_true))
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(eta_true[:, 1], eta_true[:, 0], label='truth trajectory')
ax.scatter(eta_measured[::10, 1], eta_measured[::10, 0], s=8, alpha=0.5, label='noisy samples')
ax.set(xlabel='East [m]', ylabel='North [m]', title='One physical trajectory, imperfect samples')
ax.axis('equal'); ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()

## Why an observer is used

The controller needs a clean low-frequency pose and body velocity. Raw position and heading measurements contain sensor noise and may also contain wave-frequency vessel motion. Direct numerical differentiation of noisy position greatly amplifies noise, so it is not a satisfactory velocity estimate.

An observer combines three sources of knowledge:

1. **Measurements:** where the vessel appears to be.
2. **A vessel model:** how pose and velocity should evolve.
3. **Known input:** the desired generalized force before thruster dynamics (`tau_est` in the assignment).

From these it estimates low-frequency pose, body velocity and, depending on the design, slowly-varying environmental bias and wave-frequency motion. Simulation truth is logged only to evaluate that estimate; it must never be fed into an enabled observer or controller.

In [ ]:
# Why differentiating the position measurement is not an observer.
# 1) With sensor noise the derivative explodes (white-noise amplification).
# 2) Even with a PERFECT sensor (noise switch off: measurement == truth),
#    the derivative returns the TOTAL velocity, dominated by the
#    wave-frequency oscillation -- not the slowly-varying velocity DP needs.
v_from_noisy = np.gradient(eta_measured[:, 0], dt)
v_from_ideal = np.gradient(eta_ideal[:, 0], dt)   # noise-free measurement

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(t, v_from_noisy, color='0.7', label='derivative of noisy measurement')
axes[0].plot(t, v_from_ideal, label='derivative of noise-free measurement')
axes[0].set(xlabel='Time [s]', ylabel='North velocity [m/s]',
            title='With sensor noise: useless')
axes[1].plot(t, v_from_ideal, label='derivative of noise-free measurement')
axes[1].set(xlabel='Time [s]',
            title='Even noise-free: wave-frequency velocity dominates')
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()

## Why an observer is needed even with ZERO sensor noise

With the noise switch off the sensor adds no noise at all — and DP still needs an observer, for two separate reasons:

1. **Velocity is not measured.** The controller needs body velocity for damping, and (above) differentiating the position measurement does not deliver it.
2. **The perfect position measurement contains first-order wave motion the controller must not chase.** The vessel oscillates metres back and forth every ~8 s. A feedback law acting on that raw signal commands thrust that reverses every few seconds — the thrusters physically cannot follow it (ramp times of seconds to full thrust, azimuth rotation at 2 rpm), so chasing the waves delivers no mean-position benefit; it only wastes power and wears out the machinery.

How to separate the low-frequency motion from the wave-frequency motion is exactly the observer design task in `part_2/observer.py` — that part is yours.

## What students should add or discuss

1. Run the mandatory simulations with ideal measurements (`use_sensor_noise=False`, the default).
2. Implement both observer templates and compare them with and without wave loads.
3. **Optional sensor-noise part:** repeat the observer comparison with `Part2SimConfig(use_sensor_noise=True)` and show how each observer copes with measurement noise on top of wave filtering. The noise levels are fixed course parameters, so results are comparable between groups — do not change them.
4. Plot true, measured and estimated states; do not judge an observer only from the controlled vessel trajectory.
5. Explain that the supplied model does not cover redundancy, voting, sensor faults, asynchronous sampling, latency, lever-arm correction, bias drift or colored noise.
6. For an advanced extension, add one omitted effect at a time—such as gyro bias, GNSS dropout or a first-order colored-noise process—and document how the observer responds.

The purpose of this simplified block is to study estimation and wave filtering clearly. It is not a certification-level DP sensor simulator.